## Vytváranie grafov použitých v aplikácií

In [19]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import os

plt.style.use("seaborn-v0_8-whitegrid")
PRIMARY = "#1e8fa4"
SECONDARY = "#32c1a8"

BASE = Path("static/plots")
(BASE / "demografia").mkdir(parents=True, exist_ok=True)
(BASE / "labs").mkdir(parents=True, exist_ok=True)
(BASE / "comorb").mkdir(parents=True, exist_ok=True)
(BASE / "drug_groups").mkdir(parents=True, exist_ok=True)


## Načítanie dát

In [ ]:
DATA_DIR = Path("data")

FILES = {
    "Wuhan": DATA_DIR / "wuhan.csv",
    "Alfa": DATA_DIR / "alfa.csv",
    "Delta": DATA_DIR / "delta.csv",
    "Omikron": DATA_DIR / "omikron.csv"
}

def read_csv_auto(path):
    return pd.read_csv(path, sep=";", encoding="utf-8", low_memory=False)

dfs = []
for wave, file in FILES.items():
    df = read_csv_auto(file)
    df["Vlna"] = wave
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data["CDI_status"] = data["A04.7"].map({0: "CDI−", 1: "CDI+"})


## Grafy pre kategóriu demografia

In [21]:
dem_folder = BASE / "demografia"

# 1) Boxplot veku
plt.figure(figsize=(6,4))
sns.boxplot(data=data, x="A04.7", y="Vek", palette=[PRIMARY, SECONDARY])
plt.xticks([0,1], ["CDI−","CDI+"])
plt.title("Vek pacientov podľa výskytu CDI")
plt.ylabel("Vek [roky]")
plt.tight_layout()
plt.savefig(dem_folder/"demografia_vek_boxplot.png", dpi=200)
plt.close()


# 2) Barplot pohlavia
percent_cdi = data.groupby("Pohlavie")["A04.7"].mean()*100
df = percent_cdi.reset_index()

plt.figure(figsize=(6,4))
sns.barplot(data=df, x="Pohlavie", y="A04.7", palette=[PRIMARY, SECONDARY])
plt.ylabel("Podiel CDI [%]")
plt.title("Výskyt CDI podľa pohlavia")
plt.tight_layout()
plt.savefig(dem_folder/"demografia_pohlavie_barplot.png", dpi=200)
plt.close()


# 3) Violinplot vek podľa vĺn
plt.figure(figsize=(8,5))
sns.violinplot(data=data, x="Vlna", y="Vek",
               hue="A04.7", split=True,
               palette=[PRIMARY, SECONDARY])
plt.title("Vekové rozloženie podľa vĺn a CDI")
plt.tight_layout()
plt.savefig(dem_folder/"demografia_vek_vlny_violin.png", dpi=200)
plt.close()


C:\Users\pavel\AppData\Local\Temp\ipykernel_2956\1336709078.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=data, x="A04.7", y="Vek", palette=[PRIMARY, SECONDARY])
C:\Users\pavel\AppData\Local\Temp\ipykernel_2956\1336709078.py:19: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df, x="Pohlavie", y="A04.7", palette=[PRIMARY, SECONDARY])


## Grafy pre kategóriu - komorbidity

In [22]:
comorb_folder = BASE / "comorb"

comorbid_cols = [
    "Hypertenzia", "Diabetes mellitus", "Kardiovaskulárne ochorenia",
    "Chronické respiračné ochorenia", "Renálne ochorenia",
    "Pečeňové ochorenia", "Onkologické ochorenia", "Imunosupresia"
]

summary = []
for c in comorbid_cols:
    pos = data.loc[data["A04.7"] == 1, c].mean()*100
    neg = data.loc[data["A04.7"] == 0, c].mean()*100
    summary.append([c, pos, neg])

df = pd.DataFrame(summary, columns=["Ochorenie", "CDI+", "CDI−"])
df_melt = df.melt(id_vars="Ochorenie", var_name="Skupina", value_name="Podiel")

plt.figure(figsize=(8,5))
sns.barplot(data=df_melt, y="Ochorenie", x="Podiel",
            hue="Skupina", palette=[PRIMARY, SECONDARY])
plt.title("Chronické ochorenia – CDI+ vs CDI−")
plt.tight_layout()
plt.savefig(comorb_folder/"comorbidity_overview.png", dpi=200)
plt.close()


## Grafy pre kategóriu - skupiny liekov 

In [23]:
drug_group_folder = BASE / "drug_groups"

drug_groups = {
    "Antivirotiká": [
        "MD652 | FABIFLU TABLETS", "MD656 IV-BECT 6MG (ivermectin)",
        "5042D | VEKLURY", "9547D | PAXLOVID", "LAGEVRIO"
    ],
    "Vitamíny": [
        "00584 | PYRIDOXIN LÉČIVA INJ", "24836 | ACIDUM ASCORBICUM BBP",
        "24814 | CALCIFEROL BBP 7,5 MG/ML", "00498 | MAGNESIUM SULFURICUM BBP 100 MG/ML INJEKČNÝ ROZTOK",
        "00449 | EREVIT 300 MG/ML", "89145 | VITAMIN C-INJEKTOPAS", "92973 ALPHA D3"
    ],
    "Imuno- (tlmenie reakcie)": [
        "02963 | PREDNISON 20 LÉČIVA", "00269 | PREDNISON 5 LÉČIVA", "84090 | DEXAMED 6",
        "1275C | DEXAMETAZÓN KRKA", "MD661 BIODEXONE-DEXAMETHASONE", "2410B HYDROCORTISONE",
        "3242C | OLUMIANT 4 MG", "Anakinra", "RoActemra"
    ],
    "Imuno+ (stimulácia)": [
        "34045 | POLYOXIDONIUM 6 MG", "87299 | IMUNOR", "56930 IMMODIN",
        "Isoprinosine, ", "3879d INOMED"
    ],
    "Antibiotiká": [
        "35715 Azithromycin", "45954 Ceftriaxon", "0471B MOLOXIN",
        "9819A MOXIFLOXACIN", "58730 CIPROFLOXACIN KABI 200", "58746 CIPROFLOXACINKABI 400"
    ],
    "Iné (PPI)": [
        "05044 OZZION", "4147C OMEMYL", "89662 NOLPAZA", "39397 PANTOPRAZOL",
        "62916 SMECTA", "30639 REASEC", "84370 LAGOSA", "93105 DEGAN "
    ],
    "Kašeľ": [
        "94918 AMBROBENE", "24859 PENTOXYPHILLINUM", "8893 ACC INJEKT",
        "24949 CODEIN ", "26846 OXANTIL"
    ],
    "Antikoagulanciá": [
        "FRAXIPARIN", "CLEXANE", "FRAGMIN", "ASPIRIN", "ANOPYRIN"
    ]
}

summary = []

for name, cols in drug_groups.items():
    cols = [c for c in cols if c in data.columns]
    pos = data.loc[data["A04.7"] == 1, cols].mean().mean()*100
    neg = data.loc[data["A04.7"] == 0, cols].mean().mean()*100
    summary.append([name, pos, neg])

df = pd.DataFrame(summary, columns=["Skupina", "CDI+", "CDI−"])
df_melt = df.melt(id_vars="Skupina", var_name="Skupina2", value_name="Podiel")

plt.figure(figsize=(8,5))
sns.barplot(data=df_melt, y="Skupina", x="Podiel",
            hue="Skupina2", palette=[PRIMARY, SECONDARY])
plt.title("Skupiny liekov – CDI+ vs CDI−")
plt.tight_layout()
plt.savefig(drug_group_folder/"drug_groups_overview.png", dpi=200)
plt.close()


In [ ]:
drug_groups_folder = BASE / "drug_groups"
drug_groups_folder.mkdir(parents=True, exist_ok=True)

for group_name, drug_list in drug_groups.items():

    existing = [c for c in drug_list if c in data.columns]

    if len(existing) == 0:
        print("Skipping:", group_name, "(no columns)")
        continue

    rows = []
    for drug in existing:
        pos = data.loc[data["A04.7"] == 1, drug].mean() * 100
        neg = data.loc[data["A04.7"] == 0, drug].mean() * 100
        rows.append([drug, "CDI+", pos])
        rows.append([drug, "CDI−", neg])

    df = pd.DataFrame(rows, columns=["Liek", "Skupina", "Podiel [%]"])

    df["sort_val"] = df.apply(
        lambda r: r["Podiel [%]"] if r["Skupina"] == "CDI+" else np.nan,
        axis=1
    )
    df = df.sort_values(["sort_val", "Liek"], ascending=False)

    plt.figure(figsize=(10, max(4, len(existing) * 0.5)))
    sns.barplot(
        data=df,
        y="Liek",
        x="Podiel [%]",
        hue="Skupina",
        palette=[PRIMARY, SECONDARY]
    )

    plt.title(group_name)
    plt.xlabel("Podiel pacientov [%]")
    plt.ylabel("Liek")
    plt.legend(title="")

    ax = plt.gca()
    for p in ax.patches:
        ax.annotate(
            f"{p.get_width():.1f}%",
            (p.get_x() + p.get_width(), p.get_y() + p.get_height()/2),
            ha="left", va="center", fontsize=9
        )

    plt.tight_layout()
    filename = group_name.replace(" ", "_").replace("/", "_")
    plt.savefig(drug_groups_folder / f"group_{filename}.png", dpi=200)
    plt.close()

    print("Saved:", group_name)


Saved: Antivirotiká
Saved: Vitamíny
Saved: Imuno- (tlmenie reakcie)
Saved: Imuno+ (stimulácia)
Saved: Antibiotiká
Saved: Iné (PPI)
Saved: Kašeľ
Saved: Antikoagulanciá


## Grafy pre kategóriu - vlny

In [ ]:
waves_folder = BASE / "waves"
waves_folder.mkdir(parents=True, exist_ok=True)

PRIMARY = "#1e8fa4"
SECONDARY = "#32c1a8"

for group_name, drug_list in drug_groups.items():

    existing = [c for c in drug_list if c in data.columns]
    if not existing:
        print("Skipping:", group_name)
        continue

    rows = []

    for wave in ["Wuhan", "Alfa", "Delta", "Omikron"]:
        df_wave = data[data["Vlna"] == wave]

        if df_wave.empty:
            continue

        df_pos = df_wave[df_wave["A04.7"] == 1]
        if not df_pos.empty:
            pos = df_pos[existing].sum(axis=1).mean()
            n_pos = len(df_pos)
        else:
            pos = 0
            n_pos = 0

        df_neg = df_wave[df_wave["A04.7"] == 0]
        if not df_neg.empty:
            neg = df_neg[existing].sum(axis=1).mean()
            n_neg = len(df_neg)
        else:
            neg = 0
            n_neg = 0

        rows.append([wave, "CDI+", pos, n_pos])
        rows.append([wave, "CDI−", neg, n_neg])

    df = pd.DataFrame(rows, columns=["Vlna", "CDI", "Počet liekov", "N"])

    if df.empty:
        print("No data:", group_name)
        continue

    plt.figure(figsize=(9, 5))

    sns.barplot(
        data=df,
        x="Vlna",
        y="Počet liekov",
        hue="CDI",
        palette=[SECONDARY, PRIMARY]
    )

    plt.title(f"{group_name} – priemerný počet liekov na pacienta podľa vĺn")
    plt.xlabel("Vlna pandémie")
    plt.ylabel("Priemerný počet liekov")
    plt.legend(title="")

    ax = plt.gca()
    for p in ax.patches:
        val = p.get_height()
        if val > 0:
            ax.annotate(
                f"{val:.2f}",
                (p.get_x() + p.get_width() / 2, val),
                ha="center",
                va="bottom",
                fontsize=9
            )

    plt.tight_layout()

    filename = group_name.replace(" ", "_").replace("/", "_")
    plt.savefig(waves_folder / f"waves_{filename}.png", dpi=200)
    plt.close()

    print("Saved counts plot:", group_name)

Saved counts plot: Antivirotiká
Saved counts plot: Vitamíny
Saved counts plot: Imuno- (tlmenie reakcie)
Saved counts plot: Imuno+ (stimulácia)
Saved counts plot: Antibiotiká
Saved counts plot: Iné (PPI)
Saved counts plot: Kašeľ
Saved counts plot: Antikoagulanciá


## Grafy pre kategóriu - laboratórne hodnoty

In [ ]:
data = pd.read_csv("data_clean_20_KNN.csv", sep=";", encoding="utf-8", low_memory=False)

labs = [
    "S-Na last",
    "S-K last",
    "HGB min",
    "S-CRP max",
    "S-CRP last",
    "S-IL6 max",
    "S-IL6 last",
    "S-IL6 min",
    "WBC max",
    "NE/LY(NLR) max",
    "NE/LY(NLR) last",
    "S-Kreat min",
    "S-Urea max",
    "S-AST min",
    "PDW max",
    "D-dimér HS max",
    "S-VITD first",
    "S-VITD max",
    "Neu abs max"
]

lab_folder = BASE / "labs"
lab_folder.mkdir(exist_ok=True)

for col in labs:
    if col not in data.columns:
        print("Skipping:", col)
        continue

    df = data[["A04.7", col]].copy()
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=["A04.7", col])

    if df.empty:
        print("No data:", col)
        continue

    low = df[col].quantile(0.01)
    high = df[col].quantile(0.99)
    df = df[(df[col] >= low) & (df[col] <= high)]

    plt.figure(figsize=(6, 4))

    sns.boxplot(
        data=df,
        x="A04.7",
        y=col,
        hue="A04.7",
        palette=[PRIMARY, SECONDARY],
        dodge=False,
        showfliers=False
    )

    plt.xticks([0, 1], ["CDI−", "CDI+"])
    plt.title(col)
    plt.xlabel("")
    plt.tight_layout()

    fname = col.replace(" ", "_").replace("/", "_")
    plt.savefig(lab_folder / f"lab_{fname}.png", dpi=200)
    plt.close()

    print("Saved:", fname)

Saved: S-Na_last
Saved: S-K_last
Saved: HGB_min
Saved: S-CRP_max
Saved: S-CRP_last
Saved: S-IL6_max
Saved: S-IL6_last
Saved: S-IL6_min
Saved: WBC_max
Saved: NE_LY(NLR)_max
Saved: NE_LY(NLR)_last
Saved: S-Kreat_min
Saved: S-Urea_max
Saved: S-AST_min
Saved: PDW_max
Saved: D-dimér_HS_max
Saved: S-VITD_first
Saved: S-VITD_max
